Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Add parent directory to path to import local modules
sys.path.insert(0, str(Path.cwd().parent))

from src.core.SEIRSParameters import*
from src.core.solvers import*
from src.core.utils import*

In [3]:
output_dir = Path.cwd().parent / "outputs" / "figures" / "nb2"
output_dir.mkdir(parents=True, exist_ok=True)

Global parameters

In [4]:
n = 50 # grid size (number of nodes in each dimension); we'll use a smaller value for quick implementtion

# diffusion rates
DS = 0.1
DE = np.exp(-9)
DI = np.exp(-9)
DR = 0.01

ALPHA = 0.01
SIGMA = 0.33

BETA_0 = 0.05
GAMMA_0 = 0.15

GAMMA_COEFF = BETA_COEFF = 0.1


POPULATION_DENSITY = 1.0 # N
OMEGA = 1.0 # space volume in 2D 
M=n*n

baseParams = SEIRSParameters(
    dS=DS,
    dE=DE,
    dI=DI,
    dR=DR,
    N=POPULATION_DENSITY,
    grid_size=n,
    alpha=ALPHA,
    sigma=SIGMA,
    beta_0=BETA_0,
    gamma_0=GAMMA_0,
    gamma_coeff=GAMMA_COEFF,
    beta_coeff=BETA_COEFF
)

In [5]:
# Gaussian parameters
R_GAMMA = 0.1 # radius/width
R_BETA = 0.15
C_GAMMA = (0.25, 0.25) # center
C_BETA = (0.75, 0.75)

In [6]:
beta_field = baseParams.beta_field(mu=C_BETA, sigma=R_BETA)

X,Y = baseParams.make_grid()
gamma_field = np.full_like(X, baseParams.gamma_0) # so gamma is constant
    
print(gamma_field)

[[0.15 0.15 0.15 ... 0.15 0.15 0.15]
 [0.15 0.15 0.15 ... 0.15 0.15 0.15]
 [0.15 0.15 0.15 ... 0.15 0.15 0.15]
 ...
 [0.15 0.15 0.15 ... 0.15 0.15 0.15]
 [0.15 0.15 0.15 ... 0.15 0.15 0.15]
 [0.15 0.15 0.15 ... 0.15 0.15 0.15]]


This makes gamma constant = gamma_0 accross all the grid.
Now let's define the other parameters.

In [ ]:
# let's vary dE and dI to tend to zero
dEI_range = np.arange(1e-5, 1e-2+1e-5, 1e-5)
dEI_range = np.arange(1.0, 50.0, 1)



R0_limit1 = (POPULATION_DENSITY/OMEGA) * np.max(np.where(gamma_field != 0, beta_field/gamma_field, 0))
R0_limit2 = (POPULATION_DENSITY/(OMEGA)) * (mass(beta_field, baseParams.h) / mass(gamma_field, baseParams.h) )

In [8]:
# we compute for each time the R0
def simulate_R0(dEI_range, r0_evolution): 

    for i in range(dEI_range.size):
        # redefine parameters; as it is froze, we create new obj each time
        params = SEIRSParameters(
            dS=DS,
            dE=dEI_range[i],
            dI=dEI_range[i],
            dR=DR,
            N=POPULATION_DENSITY,
            grid_size=n,
            alpha=ALPHA,
            sigma=SIGMA,
            beta_0=BETA_0,
            gamma_0=GAMMA_0,
            gamma_coeff=GAMMA_COEFF,
            beta_coeff=BETA_COEFF
        )
        beta_field_temp = params.beta_field(C_BETA, R_BETA)
        gamma_field_temp = params.gamma_field(C_GAMMA, R_GAMMA)
        mu_star_temp = compute_mu_star(params=params, beta_field=beta_field_temp, gamma_field=gamma_field_temp)
        r0_temp = params.N/(OMEGA*mu_star_temp)
        r0_evolution[i] = r0_temp
    
    return dEI_range, r0_evolution
    
def plot_R0_limit(dEI_range, r0_evolution, r0_limit,
                      line_label='r0 evolution', limit_label='Limit',
                      title='R0 limit', xlabel='dE, dI', ylabel='R0',
                      ax=None, show=True, save_path=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    else:
        fig = ax.figure

    sb.lineplot(x=dEI_range, y=r0_evolution, ax=ax,
                 label=line_label, linewidth=2.2)
    ax.axhline(r0_limit, color='red', linestyle='--',
               linewidth=1.5, label=limit_label)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    if save_path:
        fig.savefig(output_dir / save_path, bbox_inches='tight')
    if show:
        plt.show()

    return fig, ax